<a href="https://colab.research.google.com/github/umeshrawat/AI_Math_Vedas/blob/master/StockPrice_Forecast.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Import necessary libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import yfinance as yf
from sklearn.preprocessing import MinMaxScaler
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, LSTM
from tensorflow.keras.optimizers import Adam

# Download GOOGL stock data
googl_data = yf.download('GOOGL', start='2020-01-01', end='2025-01-25')

# Preprocess the data
df = googl_data[['Close']]
scaler = MinMaxScaler()
scaled_data = scaler.fit_transform(df)

# Create sequences for training
def create_sequences(data, seq_length):
    X, y = [], []
    for i in range(len(data) - seq_length):
        X.append(data[i:i+seq_length])
        y.append(data[i+seq_length])
    return np.array(X), np.array(y)

seq_length = 60
X, y = create_sequences(scaled_data, seq_length)

# Split data into training and testing sets
train_size = int(len(X) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

# Build the Deep Q-Learning model
model = Sequential([
    LSTM(50, return_sequences=True, input_shape=(seq_length, 1)),
    LSTM(50, return_sequences=False),
    Dense(25),
    Dense(1)
])

model.compile(optimizer=Adam(learning_rate=0.001), loss='mean_squared_error')

# Train the model
history = model.fit(X_train, y_train, batch_size=32, epochs=100, validation_split=0.1, verbose=1)

# Evaluate the model
train_loss = model.evaluate(X_train, y_train, verbose=0)
test_loss = model.evaluate(X_test, y_test, verbose=0)
print(f'Train Loss: {train_loss}')
print(f'Test Loss: {test_loss}')

# Make predictions
predictions = model.predict(X_test)
predictions = scaler.inverse_transform(predictions)
y_test_inv = scaler.inverse_transform(y_test)

# Visualize the results
plt.figure(figsize=(12, 6))
plt.plot(y_test_inv, label='Actual Price')
plt.plot(predictions, label='Predicted Price')
plt.title('GOOGL Stock Price Prediction')
plt.xlabel('Time')
plt.ylabel('Price')
plt.legend()
plt.show()

# Implement a simple trading strategy
def trading_strategy(actual, predicted):
    positions = np.where(predicted > actual, 1, -1)
    returns = positions[:-1] * np.diff(actual)
    return np.sum(returns)

total_returns = trading_strategy(y_test_inv, predictions)
print(f'Total Returns: ${total_returns:.2f}')

# Implement Deep Q-Learning for trading decisions
class TradingEnvironment:
    def __init__(self, data):
        self.data = data
        self.current_step = 0

    def reset(self):
        self.current_step = 0
        return self.data[self.current_step]

    def step(self, action):
        self.current_step += 1
        if self.current_step >= len(self.data) - 1:
            done = True
        else:
            done = False

        reward = self.calculate_reward(action)
        next_state = self.data[self.current_step]

        return next_state, reward, done

    def calculate_reward(self, action):
        current_price = self.data[self.current_step-1]
        next_price = self.data[self.current_step]
        if action == 1:  # Buy
            return (next_price - current_price) / current_price
        elif action == 0:  # Hold
            return 0
        else:  # Sell
            return (current_price - next_price) / current_price

# Create DQN agent
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense
from tensorflow.keras.optimizers import Adam
from collections import deque
import random

class DQNAgent:
    def __init__(self, state_size, action_size):
        self.state_size = state_size
        self.action_size = action_size
        self.memory = deque(maxlen=2000)
        self.gamma = 0.95
        self.epsilon = 1.0
        self.epsilon_min = 0.01
        self.epsilon_decay = 0.995
        self.learning_rate = 0.001
        self.model = self._build_model()

    def _build_model(self):
        model = Sequential([
            Dense(24, input_dim=self.state_size, activation='relu'),
            Dense(24, activation='relu'),
            Dense(self.action_size, activation='linear')
        ])
        model.compile(loss='mse', optimizer=Adam(learning_rate=self.learning_rate))
        return model

    def remember(self, state, action, reward, next_state, done):
        self.memory.append((state, action, reward, next_state, done))

    def act(self, state):
        if np.random.rand() <= self.epsilon:
            return random.randrange(self.action_size)
        act_values = self.model.predict(state)
        return np.argmax(act_values[0])

    def replay(self, batch_size):
        minibatch = random.sample(self.memory, batch_size)
        for state, action, reward, next_state, done in minibatch:
            target = reward
            if not done:
                target = reward + self.gamma * np.amax(self.model.predict(next_state)[0])
            target_f = self.model.predict(state)
            target_f[0][action] = target
            self.model.fit(state, target_f, epochs=1, verbose=0)
        if self.epsilon > self.epsilon_min:
            self.epsilon *= self.epsilon_decay

# Train the DQN agent
env = TradingEnvironment(scaled_data)
state_size = 1
action_size = 3  # Buy, Hold, Sell
agent = DQNAgent(state_size, action_size)

episodes = 100
batch_size = 32

for e in range(episodes):
    state = env.reset()
    state = np.reshape(state, [1, state_size])
    for time in range(len(scaled_data) - 1):
        action = agent.act(state)
        next_state, reward, done = env.step(action)
        next_state = np.reshape(next_state, [1, state_size])
        agent.remember(state, action, reward, next_state, done)
        state = next_state
        if done:
            print(f"Episode: {e+1}/{episodes}, Score: {time}")
            break
        if len(agent.memory) > batch_size:
            agent.replay(batch_size)

# Test the trained agent
test_data = scaled_data[-100:]
env = TradingEnvironment(test_data)
state = env.reset()
state = np.reshape(state, [1, state_size])
done = False
actions = []

while not done:
    action = agent.act(state)
    actions.append(action)
    next_state, reward, done = env.step(action)
    state = np.reshape(next_state, [1, state_size])

# Visualize the trading decisions
plt.figure(figsize=(12, 6))
plt.plot(scaler.inverse_transform(test_data), label='Stock Price')
buy_points = [i for i in range(len(actions)) if actions[i] == 0]
sell_points = [i for i in range(len(actions)) if actions[i] == 2]
plt.scatter(buy_points, scaler.inverse_transform(test_data)[buy_points], color='green', label='Buy', marker='^')
plt.scatter(sell_points, scaler.inverse_transform(test_data)[sell_points], color='red', label='Sell', marker='v')
plt.title('GOOGL Stock Trading Decisions')
plt.xlabel('Time')
plt.ylabel('Price')
plt.legend()
plt.show()

# Calculate returns
initial_balance = 10000
balance = initial_balance
shares = 0

for i, action in enumerate(actions):
    price = scaler.inverse_transform(test_data)[i][0]
    if action == 0 and balance > price:  # Buy
        shares_to_buy = balance // price
        shares += shares_to_buy
        balance -= shares_to_buy * price
    elif action == 2 and shares > 0:  # Sell
        balance += shares * price
        shares = 0

final_balance = balance + shares * scaler.inverse_transform(test_data)[-1][0]
returns = (final_balance - initial_balance) / initial_balance * 100

print(f"Initial Balance: ${initial_balance:.2f}")
print(f"Final Balance: ${final_balance:.2f}")
print(f"Returns: {returns:.2f}%")


YF.download() has changed argument auto_adjust default to True


[*********************100%***********************]  1 of 1 completed
ERROR:yfinance:
1 Failed download:
ERROR:yfinance:['FCB']: YFPricesMissingError('possibly delisted; no price data found  (1d 2020-01-01 -> 2025-01-25)')


ValueError: Found array with 0 sample(s) (shape=(0, 1)) while a minimum of 1 is required by MinMaxScaler.